<a href="https://colab.research.google.com/github/MarcAmil30/DOGMA/blob/main/boltz2_proto_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install git+https://github.com/evo-design/proto-tools.git

  Cloning https://github.com/evo-design/proto-tools.git to /tmp/pip-req-build-1pxldjcj
  Running command git clone --filter=blob:none --quiet https://github.com/evo-design/proto-tools.git /tmp/pip-req-build-1pxldjcj
  Resolved https://github.com/evo-design/proto-tools.git to commit cd417b48e4c1e4a6869297ea5ff3e034ae0625c4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.4 MB/s eta 0:0

In [2]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("boltz2")
display_overview("boltz2")
display_docs_section("boltz2", "Background")

# Boltz-2

Boltz-2 is an openly licensed biomolecular structure prediction model from the [MIT Jameel Clinic](https://jclinic.mit.edu/) and [Recursion](https://www.recursion.com/), built in the AlphaFold3 family: a diffusion model that predicts the joint 3D structure of complexes mixing proteins, DNA, RNA, and small-molecule ligands, together with the binding affinity of a small molecule against a protein target. This toolkit runs Boltz-2 structure and affinity prediction on a local GPU, with optional ColabFold multiple-sequence alignments.

Boltz-2 ([Passaro et al., 2025](https://doi.org/10.1101/2025.06.14.659707)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It builds on Boltz-1, one of the most widely used open-source alternatives to AlphaFold3, extending that co-folding model with a binding-affinity module, improved controllability, and additional training data. Like AlphaFold3, a single model folds complexes that mix proteins, DNA, RNA, and small-molecule ligands and predicts how those components are arranged relative to one another. Each protein chain can be paired with a multiple-sequence alignment (MSA) of evolutionarily related sequences, whose covariation patterns supply the evolutionary signal the model uses to place residues.

Architecturally, Boltz-2 reproduces AlphaFold3: it carries a single representation of the input tokens and a pairwise representation over token pairs, refines them through an AlphaFold3-style trunk, and generates all-atom coordinates with a diffusion module that starts from noise and iteratively denoises into a structure. Several structures can be sampled per complex and ranked by a confidence score, reported as a complex predicted local distance difference test (pLDDT) for local reliability, a predicted aligned error (PAE) for the relative placement of any two tokens, and predicted template-modeling (pTM) and interface predicted template-modeling (ipTM) scores that summarize overall and interface accuracy. Beyond structure, Boltz-2 adds a binding-affinity module that approaches the accuracy of physics-based free-energy perturbation while running more than 1000 times faster.

The reference implementation is open-sourced at [jwohlwend/boltz](https://github.com/jwohlwend/boltz) under the MIT license, covering the code, weights, and training pipeline for both academic and commercial use, with the released weights distributed as `boltz-community/boltz-2`. It was developed by the Boltz team at the [MIT Jameel Clinic](https://jclinic.mit.edu/) together with [Recursion](https://www.recursion.com/).

In [3]:
display_available_tools("boltz2")

- **`run_boltz2_affinity()`** — Predicted binding affinity (log10 IC50 μM) and binder probability for a small molecule against a protein target, via Boltz-2.
- **`run_boltz2()`** — Multi-modal structure prediction using Boltz2

In [4]:
from pathlib import Path

from proto_tools import (
    Mmseqs2HomologySearchConfig,
    Boltz2Config,
    Boltz2Input,
    Chain,
    Complex,
    run_boltz2,
)

In [5]:
# Display input docs
display_api_reference("boltz2", "input", "run_boltz2")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Boltz2Input(complexes=[complex])

**Input** — `Boltz2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>complexes</code> | <code>list[Complex]</code> | required | List of complexes to predict structure for containing chains and entity types. |
| <code>msas</code> | <code>list[ComplexMSAs] &#124; None</code> | <code>None</code> | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [6]:
# Display config docs
display_api_reference("boltz2", "config", "run_boltz2")

# Configure Boltz-2 with reduced settings for a fast demonstration run
config = Boltz2Config(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=True,
    msa_search_config=Mmseqs2HomologySearchConfig(search_mode="remote"),
    recycling_steps=3,
    diffusion_samples=1,
)

**Config** — `Boltz2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on (e.g., 'cuda', 'cpu') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1200</code> | Maximum execution time in seconds. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>include_pae_matrix</code> | <code>bool</code> | <code>False</code> | Attach the full per-residue PAE matrix. |
| <code>use_msa</code> | <code>bool</code> | <code>True</code> | Whether to auto-generate MSAs via MMseqs2 homology search; supplied MSAs are always used |
| <code>msa_search_config</code> | <code>Mmseqs2HomologySearchConfig &#124; None</code> | <code>None</code> | Nested MMseqs2 homology-search config for MSA generation; None uses default settings. |
| <code>pair_heterocomplex_msas</code> | <code>bool</code> | <code>True</code> | Whether heterocomplex protein chains should use taxonomy-paired MSA generation. |
| <code>recycling_steps</code> | <code>int</code> | <code>3</code> | Iterative refinement passes through the model. Higher = more accurate but slower. |
| <code>sampling_steps</code> | <code>int</code> | <code>200</code> | Denoising steps in the diffusion process. Higher = more refined but slower. |
| <code>diffusion_samples</code> | <code>int</code> | <code>1</code> | Structure samples per complex; best by confidence is kept. Higher = more thorough but slower. |
| <code>step_scale</code> | <code>float</code> | <code>1.5</code> | Diffusion step size (typical range 1.0-2.0). Lower values produce more sample diversity. |
| <code>max_msa_seqs</code> | <code>int</code> | <code>8192</code> | Maximum number of MSA sequences fed into the model. Lower this to reduce GPU memory on deep MSAs. |
| <code>subsample_msa</code> | <code>bool</code> | <code>False</code> | Randomly subsample the MSA on each run for sample diversity (loses determinism). |
| <code>num_workers</code> | <code>int</code> | <code>2</code> | Number of dataloader workers for prediction. |

In [7]:
# Run structure prediction
result = run_boltz2(inputs, config)

Streaming output truncated to the last 5000 lines.


In [8]:
# Display output docs
display_api_reference("boltz2", "output", "run_boltz2")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains:  {len(complex.chains)}")
print(f"  Protein length:    {len(mfng_sequence)} residues")
print(f"  Confidence score:  {mfng_structure.metrics.confidence_score:.3f}")
print(f"  Complex pLDDT:     {mfng_structure.metrics.complex_plddt:.3f}")
print(f"  pTM score:         {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score:        {mfng_structure.metrics.iptm:.3f}")
print(f"  Ligand ipTM:       {mfng_structure.metrics.ligand_iptm:.3f}")

**Output** — `Boltz2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[Structure]</code> | required | List of predicted structures, one per input complex. |

**Metrics** — `Boltz2Metrics` (one set per `structures` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>confidence_score</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>iptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>chains_ptm</code> | <code>list[float]</code> | <code>[0, 1]</code> |  |  |
| <code>pair_chains_iptm</code> | <code>list[list[float]]</code> | <code>[0, 1]</code> |  |  |
| <code>avg_pae</code> | <code>float</code> | <code>[0, 32]</code> |  |  |
| <code>pae</code> | <code>list[list[float]]</code> | <code>[0, 32]</code> |  |  |
| <code>ligand_iptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>protein_iptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>complex_plddt</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>complex_iplddt</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>complex_pde</code> | <code>float</code> | <code>&gt;= 0</code> |  |  |
| <code>complex_ipde</code> | <code>float</code> | <code>&gt;= 0</code> |  |  |

  Number of chains:  2
  Protein length:    384 residues
  Confidence score:  0.914
  Complex pLDDT:     0.929
  pTM score:         0.922
  ipTM score:        0.856
  Ligand ipTM:       0.856


In [9]:

mfng_structure.visualize(style="cartoon", color_by="bfactor")